# Interval logic examples

This notebook introduces the immutable formula API, absolute-time-set semantics, forward-looking hold formulas, compact simplification, explicit normalization, and custom derived operators. Intervals come from [`portion`](https://github.com/AlexandreDecan/portion).

## Logic and semantics

Fix a time universe $U\subseteq\mathbb R$ and let $\mathbb N_0=\{0,1,2,\ldots\}$. Every formula denotes the subset of $U$ where it is true.

### Signals and atoms

A numeric signal is a piecewise-constant function $j:U\to\mathbb N_0$. A numeric term is either a literal or a signal:

$$\tau ::= n \mid j, \qquad n\in\mathbb N_0.$$

Atomic formulas are propositions or equalities between numeric terms:

$$a ::= p \mid (\tau_1=\tau_2).$$

A proposition $p$ has a truth set $V(p)\subseteq U$. Time is implicit in an equality: its signals are read at the time being evaluated. Thus

$$
\begin{aligned}
\llbracket p\rrbracket &= V(p),\\
\llbracket\tau_1=\tau_2\rrbracket &= \{t\in U\mid \tau_1(t)=\tau_2(t)\}.
\end{aligned}
$$

A literal is constant, so $n(t)=n$. Inequality is shorthand: $\tau_1\neq\tau_2:=\neg(\tau_1=\tau_2)$.

### Formulas

Using an interval $I$, the minimal language is

$$\varphi ::= \top \mid a \mid \neg\varphi \mid (\varphi\land\varphi) \mid I_\varphi \mid \exists_I\varphi \mid H^\delta\varphi, \qquad \delta\in\mathbb R_{\geq 0}.$$

The remaining forms are abbreviations:

$$
\begin{aligned}
\bot &:= \neg\top, &
\varphi\lor\psi &:= \neg(\neg\varphi\land\neg\psi),\\
\varphi\rightarrow\psi &:= \neg\varphi\lor\psi, &
I^{\varphi} &:= \neg I_{\neg\varphi},\\
\forall_I\varphi &:= \neg\exists_I\neg\varphi.
\end{aligned}
$$

The implementation may keep optimized nodes for these derived forms.

### Boolean semantics

The Boolean clauses remain the usual set operations:

$$
\begin{aligned}
\llbracket\top\rrbracket &= U, & \llbracket\bot\rrbracket &= \varnothing, \\
\llbracket\neg\varphi\rrbracket &= U\setminus\llbracket\varphi\rrbracket, &
\llbracket\varphi\land\psi\rrbracket &= \llbracket\varphi\rrbracket\cap\llbracket\psi\rrbracket, \\
\llbracket\varphi\lor\psi\rrbracket &= \llbracket\varphi\rrbracket\cup\llbracket\psi\rrbracket.
\end{aligned}
$$

### Time windows

The formula $I_\top$ is the characteristic formula of $I\cap U$. A restrictive window is false outside $I$ and follows $\varphi$ inside it; its vacuous dual is true outside $I$ and follows $\varphi$ inside it:

$$
\begin{aligned}
\llbracket I_\top\rrbracket &= U\cap I,\\
\llbracket I_\varphi\rrbracket &= I\cap\llbracket\varphi\rrbracket,\\
\llbracket I^{\varphi}\rrbracket &= (U\setminus I)\cup\llbracket I_\varphi\rrbracket.
\end{aligned}
$$

Equivalently, $I_\varphi\equiv I_\top\land\varphi$ and $I^{\varphi}\equiv I_\top\rightarrow\varphi$.

### Hold from the current time

The forward-looking formula $H^\delta\varphi$ requires $\varphi$ throughout the half-open interval of length $\delta$ that starts now. Since evaluation is restricted to the explicit universe, its denotation is

$$
\llbracket H^\delta\varphi\rrbracket
=\{t\in U\mid [t,t+\delta)\cap U\subseteq\llbracket\varphi\rrbracket\}.
$$

The left endpoint is included and the right endpoint is excluded: failure at time $t$ violates the hold, while failure exactly at $t+\delta$ does not. The empty window makes $H^0\varphi\equiv\top$. For $\delta>0$, when $[a,b+\delta)\subseteq U$,

$$\forall_{[a,b]}H^\delta\varphi\equiv\forall_{[a,b+\delta)}\varphi.$$

### Interval quantification

Time quantification ranges over $I\cap U$ and produces a global truth value:

$$
\llbracket\forall_I\varphi\rrbracket=
\begin{cases}
U & I\cap U\subseteq\llbracket\varphi\rrbracket,\\
\varnothing & \text{otherwise,}
\end{cases}
\qquad
\llbracket\exists_I\varphi\rrbracket=
\begin{cases}
U & I\cap\llbracket\varphi\rrbracket\neq\varnothing,\\
\varnothing & \text{otherwise.}
\end{cases}
$$

In particular, $\forall_\varnothing\varphi\equiv\top$ and $\exists_\varnothing\varphi\equiv\bot$. Pointwise satisfaction is induced by denotation:

$$t\models\varphi \quad\Longleftrightarrow\quad t\in\llbracket\varphi\rrbracket.$$

In negation normal form, negations stop at propositions, equalities, or a `Hold` node whose existential temporal dual is not part of the language.

### Charging-station example

Consider one charging station with two charging points. Let

$$j_1,j_2:U\to\mathbb N_0$$

be their job-allocation signals, where $0$ means *no job* and each positive number is a job ID. In formulas, time is implicit: $j_1=0$ is an atomic equality whose denotation is

$$\llbracket j_1=0\rrbracket=\{t\in U\mid j_1(t)=0\}.$$

Thus $t\models j_1=0$ says exactly that charging point 1 is unallocated at time $t$. The temporal operators can then express allocation requirements directly. For example,

$$
\begin{aligned}
\exists_I(j_1=0)
&\quad\text{means point 1 is free sometime during }I,\\
\forall_I((j_1=0)\land(j_2=0))
&\quad\text{means both points are free throughout }I,\\
\forall_I\bigl((j_1\neq j_2)\lor((j_1=0)\land(j_2=0))\bigr)
&\quad\text{forbids assigning the same nonzero job to both points during }I.
\end{aligned}
$$

`denote(φ, V, U, signals=J)` computes $\llbracket\varphi\rrbracket$ and `holds(φ, V, t, U, signals=J)` tests membership in that truth set. Proposition valuations and piecewise-constant natural-number signals are clipped to the explicit universe. Open, closed, disconnected, empty, and unbounded intervals retain their exact set-theoretic meaning.

In [1]:
import portion as P

from interval_logic import (
    And, Atom, Bool, DerivedExpression, Equal, Equivalent, Exists, ForAll,
    Guard, Hold, Implies, NaturalSignal, Not, Or, Restrict, Signal, Xor,
    denote, holds, parse, partition_intervals,
    simplify, timeline, to_nnf,
)

## 1. Construct formulas

Atoms describe facts that may hold at different absolute times. Use `~`, `&`, and `|` for negation, conjunction, and disjunction. The explicit `And` and `Or` nodes accept any number of operands.

The first formula below is

$$c \land (a \lor h),$$

where $c$, $a$, and $h$ mean `connected`, `available`, and `charged`. Parentheses matter because conjunction binds more tightly than disjunction in the pretty printer. `depth` is the longest root-to-leaf path, and `node_count` counts every AST occurrence.

In [2]:
available = Atom("available")
charged = Atom("charged")
connected = Atom("connected")

pointwise_requirement = connected & (available | charged)
pointwise_requirement

And(operands=(Atom(name='connected'), Or(operands=(Atom(name='available'), Atom(name='charged')))))

In [3]:
print("depth:", pointwise_requirement.depth)
print("nodes:", pointwise_requirement.node_count)
print("immutable/hashable:", hash(pointwise_requirement))

depth: 3
nodes: 5
immutable/hashable: -3611626946907674874


## 2. Evaluate formulas over time

A valuation maps every atom name to the set of absolute times where it is true. `denote` returns the part of the universe where the complete formula holds; `holds` answers the same question at one instant.

For the formula $c \land (a \lor h)$, evaluation is ordinary set algebra:

$$\llbracket c \land (a \lor h)\rrbracket = C \cap (A \cup H).$$

Here $A \cup H = [8,18]$ and $C=[9,17]$, hence the displayed truth set is $[9,17]$. Therefore the formula holds at 11 but not at 18.

In [4]:
universe = P.closed(0, 24)
valuation = {
    "available": P.closedopen(8, 12) | P.closed(14, 18),
    "charged": P.closed(10, 16),
    "connected": P.closed(9, 17),
}

truth_times = denote(pointwise_requirement, valuation, universe)
print("truth set:", truth_times)
print("holds at 11:", holds(pointwise_requirement, valuation, 11, universe))
print("holds at 18:", holds(pointwise_requirement, valuation, 18, universe))

truth set: [9,17]
holds at 11: True
holds at 18: False


### Visualize the valuation and denotation

`timeline` places the exact truth set of each valuation row beside the formula's denotation. The blue bars are atom valuations, the gray bar is $U$, and the green bar is $\llbracket\varphi\rrbracket_V$. Filled endpoint circles are closed; hollow circles are open.

For this example the green row visualizes the same calculation as above:

$$\llbracket\varphi\rrbracket_V = C \cap (A \cup H) = [9,17].$$

The table beneath the chart is exact; the horizontal scale is only a visual projection of those sets onto the finite numeric universe.

In [5]:
formula_timeline = timeline(
    pointwise_requirement,
    valuation,
    universe,
    formula_label="φ = connected ∧ (available ∨ charged)",
)
formula_timeline

Row,Holds on
Universe,"[0,24]"
available,"[8,12) | [14,18]"
charged,"[10,16]"
connected,"[9,17]"
φ = connected ∧ (available ∨ charged),"[9,17]"


### Restrictive and vacuous time windows

A restrictive window $I_\varphi$ keeps the truth of $\varphi$ inside $I$ and is false outside it. Its guarded dual $I^{\varphi}$ also follows $\varphi$ inside $I$, but is vacuously true outside it:

$$
\llbracket I_\varphi\rrbracket = I \cap \llbracket\varphi\rrbracket,
\qquad
\llbracket I^{\varphi}\rrbracket = (U\setminus I) \cup \llbracket I_\varphi\rrbracket.
$$

`Restrict(I, φ)` and `φ.restrict(I)` construct the first form; `Guard(I, φ)` and `φ.guard(I)` construct the second. The following timelines make the different outside-window defaults visible.

In [6]:
service_window = P.closedopen(10, 14)
restricted_availability = available.restrict(service_window)

print("restrictive:", restricted_availability)
timeline(restricted_availability, valuation, universe)

restrictive: [10,14)_(available)


Row,Holds on
Universe,"[0,24]"
available,"[8,12) | [14,18]"
charged,"[10,16]"
connected,"[9,17]"
"φ = [10,14)_(available)","[10,12)"


In [7]:
guarded_availability = available.guard(service_window)

print("guarded:", guarded_availability)
timeline(guarded_availability, valuation, universe)

guarded: [10,14)^(available)


Row,Holds on
Universe,"[0,24]"
available,"[8,12) | [14,18]"
charged,"[10,16]"
connected,"[9,17]"
"φ = [10,14)^(available)","[0,12) | [14,24]"


### Hold from now

`Hold(delta, body)`, `body.hold(delta)`, and the parsed notation `H^delta body` construct the same forward-looking formula. For the current valuation, `connected` holds on $[9,17]$. Consequently $H^2c$ holds on $[9,15]$: at time $15$ its required window is $[15,17)$, but any later start reaches times after $17$ where `connected` is false.

The assertion below also demonstrates the quantifier law. Universal quantification over every start time in $[9,12]$, followed by a two-hour hold, covers exactly $[9,14)$ because hold windows exclude their right endpoints.

In [ ]:
two_hour_connection = Hold(2, connected)
assert two_hour_connection == connected.hold(2)
assert two_hour_connection == parse("H^2 connected")

start_times = P.closed(9, 12)
covered_times = P.closedopen(9, 14)
starts_with_hold = ForAll(start_times, two_hour_connection)
extended_forall = ForAll(covered_times, connected)
assert denote(starts_with_hold, valuation, universe) == denote(extended_forall, valuation, universe)

print("formula:", two_hour_connection)
print("truth set:", denote(two_hour_connection, valuation, universe))
print("holds at 15:", holds(two_hour_connection, valuation, 15, universe))
print("holds just after 15:", holds(two_hour_connection, valuation, 15.1, universe))
print("quantified equivalence:", denote(starts_with_hold, valuation, universe))

formula: H^2 connected
truth set: [9,15.0]
holds at 15: True
holds just after 15: False
quantified equivalence: [0,24]


## 3. Quantify over fixed intervals

`ForAll(I, body)` is globally true when the body holds everywhere in $I \cap U$. `Exists(I, body)` is globally true when it holds somewhere in that interval. A true quantified sentence therefore denotes the entire universe; a false one denotes the empty set.

In this example, $[9,12) \subseteq C=[9,17]$, so $\forall_{[9,12)}c$ is true. Also $[12,17] \cap H=[12,16] \neq \varnothing$, so $\exists_{[12,17]}h$ is true. Their conjunction is consequently true and denotes all of $U=[0,24]$.

In [8]:
morning = P.closedopen(9, 12)
afternoon = P.closed(12, 17)

always_connected_morning = ForAll(morning, connected)
charged_sometime_afternoon = Exists(afternoon, charged)
schedule_ok = always_connected_morning & charged_sometime_afternoon

print("always connected:", denote(always_connected_morning, valuation, universe))
print("charged sometime:", denote(charged_sometime_afternoon, valuation, universe))
print("schedule valid:", denote(schedule_ok, valuation, universe))
timeline(schedule_ok, valuation, universe, formula_label=" ")

always connected: [0,24]
charged sometime: [0,24]
schedule valid: [0,24]


Row,Holds on
Universe,"[0,24]"
available,"[8,12) | [14,18]"
charged,"[10,16]"
connected,"[9,17]"
,"[0,24]"


The empty-domain results follow directly from the identities

$$\varnothing \subseteq S \quad\text{and}\quad \varnothing \cap S = \varnothing$$

for every truth set $S$. The evaluator therefore does not need an atom valuation to decide either empty quantifier.

In [9]:
# Standard vacuous truth over an empty quantified domain.
print("forall empty:", denote(ForAll(P.empty(), available), {}, universe))
print("exists empty:", denote(Exists(P.empty(), available), {}, universe))

forall empty: [0,24]
exists empty: ()


## 4. Charging-station allocation

Consider one station with two charging points. The signals $j_1,j_2:U\to\mathbb N_0$ contain the job ID assigned to each point; $0$ means that the point is idle. `NaturalSignal` stores each non-default value on an exact Portion interval, while `default=0` fills the remaining universe.

A valid allocation never assigns the same nonzero job to both points. Define the pointwise safety formula

$$\chi := (j_1\neq j_2)\lor((j_1=0)\land(j_2=0)).$$

The station is valid throughout the planning horizon exactly when $\forall_U\chi$ holds. In Python, `.equals()` and `.not_equals()` construct the logical comparisons; structural `==` and `!=` must not be used to build formulas.

In [10]:
station_horizon = P.closed(0, 12)
j_1, j_2 = Signal("j_1"), Signal("j_2")
job_signals = {
    "j_1": NaturalSignal({
        11: P.closedopen(1, 4),
        22: P.closedopen(6, 9),
    }, default=0),
    "j_2": NaturalSignal({
        33: P.closedopen(2, 5),
        22: P.closedopen(8, 10),  # conflicts with point 1 on [8,9)
    }, default=0),
}

allocation_safe = j_1.not_equals(j_2) | (j_1.equals(0) & j_2.equals(0))
station_valid = ForAll(station_horizon, allocation_safe)

print("safe times:    ", denote(allocation_safe, {}, station_horizon, signals=job_signals))
print("globally valid:", denote(station_valid, {}, station_horizon, signals=job_signals))

safe times:     [0,8) | [9,12]
globally valid: ()


Job 22 occupies both charging points on $[8,9)$, hence

$$\llbracket\chi\rrbracket=[0,8)\cup[9,12],\qquad \llbracket\forall_U\chi\rrbracket=\varnothing.$$

The orange timeline rows show the value regions of $j_1$ and $j_2$; the green row shows where $\chi$ holds. This makes the conflicting interval visible without expanding every job ID into a separate proposition.

In [11]:
timeline(
    allocation_safe,
    {},
    station_horizon,
    signals=job_signals,
    formula_label="χ = allocation safe",
)

Row,Holds on
Universe,"[0,12]"
j_1 = 0,"[0,1) | [4,6) | [9,12]"
j_1 = 11,"[1,4)"
j_1 = 22,"[6,9)"
j_2 = 0,"[0,2) | [5,8) | [10,12]"
j_2 = 22,"[8,10)"
j_2 = 33,"[2,5)"
χ = allocation safe,"[0,8) | [9,12]"


## 5. Compact simplification

The default simplifier applies size-nonincreasing equivalences, canonicalizes commutative operands, and reaches a deterministic fixed point. The first example uses

$$\top \land a = a, \qquad a \land a = a, \qquad a \land (a \lor h) = a,$$

where the final equality is the absorption law. Together they reduce $\top \land a \land a \land (a \lor h)$ to $a$. Idempotence means $s(s(\varphi))=s(\varphi)$.

In [12]:
noisy = And((
    Bool(True),
    available,
    available,
    Or((available, charged)),  # absorbed by `available`
))
compact = simplify(noisy)
print("before:", noisy)
print("after: ", compact)
assert simplify(compact) == compact

before: ⊤ ∧ available ∧ available ∧ (available ∨ charged)
after:  available


The second example applies two quantifier laws:

$$
(\forall_I\varphi) \land (\forall_I\psi) \equiv \forall_I(\varphi \land \psi),
$$

$$
(\exists_I\varphi) \lor (\exists_J\varphi) \equiv \exists_{I\cup J}\varphi.
$$

The dual merge laws are intentionally not assumed: existential quantifiers do not generally distribute over conjunction, and universal quantifiers do not generally distribute over disjunction.

In [13]:
same_window = And((
    ForAll(morning, available),
    ForAll(morning, connected),
))
same_body = Or((
    Exists(P.closed(8, 10), charged),
    Exists(P.closed(10, 12), charged),
))

print("combine bodies:   ", simplify(same_window))
print("combine intervals:", simplify(same_body))

combine bodies:    ∀[9,12): available ∧ connected
combine intervals: ∃[8,12]: charged


## 6. Derived Boolean operators and negation normal form

Derived nodes remain readable when constructed, then lower into the trusted core language:

$$
\varphi \to \psi \equiv \neg\varphi \lor \psi, \qquad
\varphi \leftrightarrow \psi \equiv (\varphi\land\psi) \lor (\neg\varphi\land\neg\psi),
$$

$$
\varphi \oplus \psi \equiv (\varphi\land\neg\psi) \lor (\neg\varphi\land\psi).
$$

`to_nnf` additionally pushes negations down using De Morgan's laws, double-negation elimination, and quantifier duality. A negated `Hold` remains `Not(Hold(...))` because the language has no existential temporal dual that could replace it.

In [14]:
policy = Implies(connected, available | charged)
agreement = Equivalent(available, charged)
exactly_one = Xor(available, charged)

print("derived policy:", policy)
print("simplified core:", simplify(policy))
print("policy at 13:", holds(policy, valuation, 13, universe))
print("equivalent at 15:", holds(agreement, valuation, 15, universe))
print("xor at 9:", holds(exactly_one, valuation, 9, universe))

derived policy: connected → available ∨ charged
simplified core: available ∨ charged ∨ ¬connected
policy at 13: True
equivalent at 15: True
xor at 9: True


For the NNF example:

$$
\begin{aligned}
\neg(c \to \forall_I a)
&\equiv \neg(\neg c \lor \forall_I a) \\
&\equiv c \land \neg\forall_I a \\
&\equiv c \land \exists_I\neg a.
\end{aligned}
$$

The last step is the duality $\neg\forall_I\varphi \equiv \exists_I\neg\varphi$.

In [ ]:
negated_requirement = Not(Implies(connected, ForAll(morning, available)))
print("original:", negated_requirement)
print("NNF:     ", to_nnf(negated_requirement))

original: ¬(connected → (∀[9,12): available))
NNF:      connected ∧ (∃[9,12): ¬available)


## 7. Partition overlapping intervals

Interval partitioning is explicit because it may increase formula size. It builds shared, disjoint cells while preserving open and closed endpoints. For $I=[8,12]$ and $J=[10,14]$, the common partition is

$$[8,10), \qquad [10,12], \qquad (12,14].$$

Universal quantification distributes over a disjoint union:

$$\forall_{A\cup B}\varphi \equiv (\forall_A\varphi) \land (\forall_B\varphi).$$

Existential quantification uses the dual disjunction law. Windows partition similarly but preserve their pointwise result:

$$
(A\cup B)_\varphi \equiv A_\varphi \lor B_\varphi,
\qquad
(A\cup B)^{\varphi} \equiv A^{\varphi} \land B^{\varphi}.
$$

`max_nodes` bounds the resulting expansion.

In [16]:
overlapping = And((
    ForAll(P.closed(8, 12), available),
    ForAll(P.closed(10, 14), connected),
))
partitioned = partition_intervals(overlapping, max_nodes=100)

print("original:   ", overlapping)
print("partitioned:", partitioned)
assert denote(overlapping, valuation, universe) == denote(partitioned, valuation, universe)

original:    (∀[8,12]: available) ∧ (∀[10,14]: connected)
partitioned: (∀[10,12]: available) ∧ (∀[10,12]: connected) ∧ (∀(12,14]: connected) ∧ (∀[8,10): available)


The assertion checks denotational equivalence,

$$\llbracket\varphi\rrbracket_V = \llbracket\operatorname{partition}(\varphi)\rrbracket_V,$$

for the current valuation. The unit suite checks the same rewrite over many independent finite models.

## 8. Add a custom derived operator

A custom operator implements `children` for structural metrics and `lower` for semantics. The example defines NAND by

$$\operatorname{NAND}(\varphi,\psi) \equiv \neg(\varphi \land \psi).$$

Because `lower()` returns exactly that core formula, evaluation and simplification need no special case for `Nand`. This is the extension contract for any future derived operator.

In [17]:
from dataclasses import dataclass
from interval_logic import Expression

@dataclass(frozen=True, slots=True)
class Nand(DerivedExpression):
    left: Expression
    right: Expression

    @property
    def children(self):
        return (self.left, self.right)

    def lower(self):
        return Not(And((self.left, self.right)))

nand_formula = Nand(available, connected)
print("custom node:", nand_formula)
print("lowered:    ", simplify(nand_formula))
print("holds at 13:", holds(nand_formula, valuation, 13, universe))

custom node: Nand(available, connected)
lowered:     ¬(available ∧ connected)
holds at 13: True


## Practical guidance

- Always pass an explicit universe when evaluating. Atom valuations, quantified intervals, and hold windows are interpreted inside it.
- Use `simplify` routinely; it is compact and idempotent.
- Use `to_nnf` to push negations through operators with logical duals; a negated `Hold` remains wrapped in `Not`.
- Use `partition_intervals` only when disjoint interval cells are useful, and keep a suitable node budget.
- Do not use formulas in Python conditionals; call `holds` or `denote` instead.